# Fontes e chaves de integração

Revisão do preliminar fornecido. Mantém o perfil da Orbe, os contratos e as referências de estorno. Remove vínculos manuais inferidos por nome e conclusões que dependiam deles. A análise bancária está no notebook 03.

In [1]:
from pathlib import Path
from decimal import Decimal
import csv, io, re, hashlib, unicodedata
from collections import Counter
from datetime import datetime
import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'docs/FORMATO_SAIDA.md').exists())
PASTA = RAIZ / 'Tarvo_Data_Pack_set2026'  # altere para outro pacote; nomes são descobertos abaixo
assert PASTA.is_dir(), f'Pacote ausente: {PASTA}'
def arquivo(padrao):
    encontrados = list(PASTA.glob(padrao))
    assert len(encontrados) == 1, (padrao, encontrados)
    return encontrados[0]
def texto(path):
    bruto = path.read_bytes()
    try: return bruto.decode('utf-8-sig'), 'utf-8-sig'
    except UnicodeDecodeError: return bruto.decode('cp1252'), 'cp1252'
def dinheiro(s):
    s = str(s).strip()
    assert re.fullmatch(r'-?(?:\d{1,3}(?:\.\d{3})+|\d+),\d{2}', s), s
    return Decimal(s.replace('.', '').replace(',', '.'))
def total(valores): return sum(valores, Decimal('0.00'))
def norm(s): return ''.join(c for c in unicodedata.normalize('NFKD', s.upper()) if not unicodedata.combining(c))
pd.set_option('display.max_colwidth', 100)


In [2]:
orbe = pd.read_csv(arquivo('01_*.csv'), sep=';', dtype=str, keep_default_na=False)
taxas = pd.read_excel(arquivo('02_*.xlsx'), sheet_name='Taxas vigentes', header=4, dtype=str).fillna('')
display(pd.DataFrame({'linhas':[len(orbe),len(taxas)],'fonte':['Orbe','Taxas (inclui rodapé)']}))
display(orbe.head())
display(taxas.head())

,linhas,fonte
0,1943,Orbe
1,27,Taxas (inclui rodapé)


,id_transacao,data_movimento,cod_estabelecimento,nome_estabelecimento,tipo_lancamento,modalidade,qtd_parcelas,bandeira,valor_bruto,taxa_aplicada_pct,valor_taxa,valor_liquido,id_referencia,data_venda_original
0,TX88410932,01/09/2026,1018325654,CASA DO ESTUDANTE VIAGENS,VENDA,CREDITO A VISTA,1,MASTERCARD,"608,88","2,69","16,37","592,51",,
1,TX88410975,01/09/2026,1018325654,CASA DO ESTUDANTE VIAGENS,VENDA,CREDITO A VISTA,1,MASTERCARD,"614,86","2,69","16,53","598,33",,
2,TX88410833,01/09/2026,1098729012,BEM-VINDO EXCHANGE,VENDA,CREDITO A VISTA,1,MASTERCARD,"834,61","2,69","22,45","812,16",,
3,TX88410128,01/09/2026,1018325654,CASA DO ESTUDANTE VIAGENS,VENDA,CREDITO PARCELADO,11,ELO,"5773,57","3,79","218,81","5554,76",,
4,TX88410316,01/09/2026,1000960410,OXFORD PREP CURSOS,VENDA,CREDITO A VISTA,1,VISA,"1645,47","2,59","42,61","1602,86",,


,Lojista,CNPJ,EC Orbe,Débito,Crédito à vista,Créd. 2-6x,Créd. 7-12x,Pix,Antecipação,Spread remessa,Vigência,Obs
0,Escola Bridge de Idiomas Ltda,22.224.579/0001-09,1075624862,0.0149,0.0259,0.0319,0.0379,0.0089,0.021,,até 14/09/2026,renegociado - ver linha de baixo
1,Bridge Idiomas (NOVA),22.224.579/0001-09,1075624862,0.0149,0.0219,0.0319,0.0379,0.0089,0.021,,a partir de 15/09/2026,só mudou o crédito à vista
2,Horizonte Viagens,78576234000179,,0.0149,0.0269,0.0319,0.0379,0.0089,0.021,,01/01/2026,
3,Kiwi Experience,07752116000126,,0.0149,0.0259,0.0319,0.0379,0.0089,0.021,,01/01/2026,EC pendente
4,Rota Global Intercâmbio Ltda,95.927.868/0001-48,1074504011,0.0149,0.0249,0.0299,0.0359,0.0089,0.021,0.015,01/01/2026,


## Integridade da Orbe

A referência de estorno permite procurar a operação original. Referência ausente neste arquivo exige histórico adicional; não prova que a operação seja inválida.

In [3]:
display(orbe['tipo_lancamento'].value_counts().to_frame('quantidade'))
print('IDs repetidos:', orbe.id_transacao.duplicated().sum())
refs = orbe[orbe.id_referencia.ne('')].copy()
refs['origem_presente'] = refs.id_referencia.isin(orbe.id_transacao)
display(refs.groupby(['tipo_lancamento','origem_presente']).size().to_frame('quantidade'))
for campo in ['valor_bruto','valor_taxa','valor_liquido']:
    orbe[campo + '_decimal'] = orbe[campo].map(dinheiro)
erros = orbe[orbe.valor_bruto_decimal - orbe.valor_taxa_decimal != orbe.valor_liquido_decimal]
print('Divergências bruto − taxa = líquido:', len(erros))
display(erros)

,quantidade
tipo_lancamento,
VENDA,1840
ESTORNO,59
ANTECIPACAO,38
REVERSAO ANTECIPACAO,6


IDs repetidos: 0


quantidade
tipo_lancamento      origem_presente            
ESTORNO              False                     9
                     True                     50
REVERSAO ANTECIPACAO True                      6

Divergências bruto − taxa = líquido: 0


,id_transacao,data_movimento,cod_estabelecimento,nome_estabelecimento,tipo_lancamento,modalidade,qtd_parcelas,bandeira,valor_bruto,taxa_aplicada_pct,valor_taxa,valor_liquido,id_referencia,data_venda_original,valor_bruto_decimal,valor_taxa_decimal,valor_liquido_decimal


## Contratos e identidade

EC e CNPJ são candidatos a chaves. Nomes servem para inspeção, sem atribuir uma identidade. Linhas sem EC, contratos conflitantes e vigências em texto precisam ser tratados antes de comparar taxas.

In [4]:
display(orbe.groupby('cod_estabelecimento').nome_estabelecimento.nunique().to_frame('nomes_por_ec'))
display(taxas[['Lojista','CNPJ','EC Orbe','Vigência','Spread remessa']])

,nomes_por_ec
cod_estabelecimento,
1000960410,1
1014537966,1
1018325654,2
1020660828,2
1023622151,2
1037647039,3
1038867654,1
1041180753,2
1043300981,1


,Lojista,CNPJ,EC Orbe,Vigência,Spread remessa
0,Escola Bridge de Idiomas Ltda,22.224.579/0001-09,1075624862,até 14/09/2026,
1,Bridge Idiomas (NOVA),22.224.579/0001-09,1075624862,a partir de 15/09/2026,
2,Horizonte Viagens,78576234000179,,01/01/2026,
3,Kiwi Experience,07752116000126,,01/01/2026,
4,Rota Global Intercâmbio Ltda,95.927.868/0001-48,1074504011,01/01/2026,0.015
5,Vela,20987545000140,1086291869,01/01/2026,
6,Andes Expedições Ltda,85694108000184,1058961395,01/01/2026,
7,Casa do Estudante Viagens Ltda,00520815000100,1018325654,01/01/2026,0.015
8,Fluente Idiomas Online Ltda,3229837000103,1066100982,01/01/2026,
9,Mundo Aberto,17.217.888/0001-11,1037647039,01/01/2026,0.012


## Consequências para as saídas

`comissoes_por_lojista.csv` e `notas_a_emitir.csv` dependem de identidade e competência verificadas. Não completar CNPJ com zeros nem inferir contrato vigente por nome. `divergencias_taxa.csv` exige taxa contratada com vigência: taxa ausente gera pendência ao comercial, não uma diferença igual a zero.

`rastro.csv` deve conservar `id_transacao` nos resultados derivados da Orbe. Uma soma financeira correta não resolve cadastro ou contrato. Próxima etapa: dimensão de lojistas com de-para aprovado, parser de percentuais e seleção de vigência sem sobreposição.